# 01 — Data Exploration
نستعرض الداتا ونفهمها قبل ما نبدأ التدريب.

**هنشتغل على:** SIIM-ACR Pneumothorax Dataset (Kaggle)

In [ ]:
# ── 1. Install & clone ──────────────────────────────────────────
!pip install -q pydicom albumentations timm grad-cam
!git clone https://github.com/YOUR_USERNAME/acute-triage-system.git
%cd acute-triage-system
!pip install -q -r requirements.txt

In [ ]:
# ── 2. Mount Google Drive (to save models) ──────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 3. Download dataset from Kaggle ────────────────────────────
# First: upload your kaggle.json to Colab Files
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle competitions download -c siim-acr-pneumothorax-segmentation -p data/raw/
!unzip -q data/raw/siim-acr-pneumothorax-segmentation.zip -d data/raw/siim/

In [ ]:
# ── 4. Explore DICOM files ──────────────────────────────────────
import pydicom, os, glob
import matplotlib.pyplot as plt
import numpy as np

dicom_files = glob.glob('data/raw/siim/**/*.dcm', recursive=True)[:6]
print(f'Found {len(dicom_files)} DICOM files (showing first 6)')

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, path in zip(axes.flat, dicom_files):
    ds  = pydicom.dcmread(path)
    img = ds.pixel_array
    ax.imshow(img, cmap='bone')
    ax.set_title(os.path.basename(path)[:20], fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5. Test our Preprocessing pipeline ─────────────────────────
import sys; sys.path.insert(0, '.')
from src.preprocessing import load_dicom_xray, get_train_transforms, get_val_transforms

sample_path = dicom_files[0]
img         = load_dicom_xray(sample_path)
print(f'Raw image shape : {img.shape}')   # (H, W, 3)
print(f'dtype           : {img.dtype}')
print(f'value range     : {img.min()} – {img.max()}')

train_tf = get_train_transforms()
tensor   = train_tf(image=img)['image']
print(f'\nAfter transforms: {tensor.shape}')  # torch.Size([3, 512, 512])
print(f'dtype           : {tensor.dtype}')

# Visualize augmented versions
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img); axes[0].set_title('Original'); axes[0].axis('off')
for i in range(1, 4):
    aug = train_tf(image=img)['image'].permute(1,2,0).numpy()
    aug = (aug - aug.min()) / (aug.max() - aug.min())  # display only
    axes[i].imshow(aug); axes[i].set_title(f'Augmented {i}'); axes[i].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# ── 6. Build a simple DataFrame for training ───────────────────
import pandas as pd

records = []
for path in glob.glob('data/raw/siim/**/*.dcm', recursive=True):
    records.append({'image_path': path, 'label': -1})  # label TBD from metadata

df = pd.DataFrame(records)
print(df.head())
print(f'Total images: {len(df)}')
df.to_csv('data/processed/xray_manifest.csv', index=False)
print('Saved manifest to data/processed/xray_manifest.csv')